## 00. Quick Start


In [1]:
print('Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.')
print('LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.')

Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.
LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.


## 01. Environment


In [2]:
import os, sys, json
from pathlib import Path
from IPython.display import display
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
AI_ROOT = next((p for p in SEARCH_ROOTS if (p / 'app').is_dir()), None)
if AI_ROOT is None: AI_ROOT = next((p / 'ai' for p in SEARCH_ROOTS if (p / 'ai' / 'app').is_dir()))
if str(AI_ROOT) not in sys.path: sys.path.insert(0, str(AI_ROOT))
print({'python': sys.version.split()[0], 'aiRoot': str(AI_ROOT)})

{'python': '3.14.5', 'aiRoot': 'C:\\Users\\seewo\\Desktop\\big_proj_01\\new_3\\ai'}


## 02. MODE


In [3]:
MODE = 'LIVE'  # MOCK | REPLAY | LIVE
RECORDINGS_DIR = AI_ROOT / 'recordings' / 'concept_portfolio_v2'
print({'mode': MODE, 'liveExternalOperationsEnabled': MODE == 'LIVE'})

{'mode': 'LIVE', 'liveExternalOperationsEnabled': True}


## 03. Environment Check


In [4]:
LIVE_ENV_KEYS = ['AI_PROVIDER', 'AI_API_KEY', 'AI_MODEL', 'MOLEG_API_KEY', 'LEGAL_REGISTRY_VERSION']
env_status = {key: bool(os.getenv(key)) for key in LIVE_ENV_KEYS}
print(env_status if MODE == 'LIVE' else {'mode': MODE, 'message': '외부 환경변수 불필요'})

{'AI_PROVIDER': True, 'AI_API_KEY': True, 'AI_MODEL': True, 'MOLEG_API_KEY': True, 'LEGAL_REGISTRY_VERSION': True}


## 04. Schema Preflight


In [5]:
from app.concept_portfolio_v2 import ConceptPortfolioEngine, ProviderGateway, ProviderMode
from app.concept_portfolio_v2.adapters import CurrentLegalAdapter
from app.concept_portfolio_v2.diagnostics.notebook_view import *
gateway = ProviderGateway(MODE, recordings_dir=RECORDINGS_DIR)
engine = ConceptPortfolioEngine(MODE, gateway=gateway)
schema_preflight = engine.schema_preflight_report()
display(show_schema_preflight(schema_preflight))
assert schema_preflight.status == 'PASS' and schema_preflight.providerCalls == 0

,스키마,상태,실패,Provider 호출
0,PlanDraftPool,PASS,[],0
1,ConceptCandidateDraft,PASS,[],0
2,SemanticDistinctnessResult,PASS,[],0
3,SemanticFidelityResult,PASS,[],0
4,SemanticArchitectureBatch,PASS,[],0
5,SemanticHypothesisBatch,PASS,[],0
6,BusinessRoleSemanticBatch,PASS,[],0
7,LegalFactDependencySemanticBatch,PASS,[],0
8,LegalFactCompletionPatch,PASS,[],0


## 05. Input


In [6]:
SCENARIO_FILE = AI_ROOT / 'fixtures' / 'concept_portfolio_v2' / 'live_scenarios.json'
SCENARIOS = {item['scenarioId']: item for item in json.loads(SCENARIO_FILE.read_text(encoding='utf-8'))}
LIVE_SCENARIO = 'CAMPUS_SECONDHAND'
LIVE_TEST_LEVEL = 'FULL_E2E'  # CORE | LEGAL_C1 | FULL_E2E | ONE_CLICK
RUN_STAGED_CORE = LIVE_TEST_LEVEL in {'CORE', 'LEGAL_C1', 'FULL_E2E'}
RUN_STAGED_LEGAL = LIVE_TEST_LEVEL in {'LEGAL_C1', 'FULL_E2E'}
RUN_STAGED_FULL = LIVE_TEST_LEVEL == 'FULL_E2E'
scenario = SCENARIOS[LIVE_SCENARIO]
TEST_INPUT = {key: scenario[key] for key in ('ideaOverview', 'problem', 'targetUsers')}
MAX_CONCEPTS = 5
display({'scenario': LIVE_SCENARIO, 'testLevel': LIVE_TEST_LEVEL, 'domain': scenario['domain'],
         'expectedStructuralFeatures': scenario['expectedStructuralFeatures']})

{'scenario': 'CAMPUS_SECONDHAND',
 'testLevel': 'FULL_E2E',
 'domain': 'Secondhand marketplace',
 'expectedStructuralFeatures': ['중개', '지역 거래', '픽업', '물리 의존']}

## 06. Idea Brief Derivation


In [7]:
seed = engine.seed_adapter.adapt(TEST_INPUT)
idea_context = None
if RUN_STAGED_CORE:
    engine._reset()
    idea_context = await engine.derive_idea_brief(seed)
print({'ideaCallComplete': bool(idea_context), 'interpretationPresent': bool(seed.interpretation),
       'stagedCore': RUN_STAGED_CORE})

{'ideaCallComplete': True, 'interpretationPresent': True, 'stagedCore': True}


## 07. Safety


In [8]:
display(idea_context.safetyReview.model_dump(mode='json') if idea_context else {'status': 'SKIPPED'})
assert idea_context is None or idea_context.safetyReview.passed

{'decision': 'ALLOW',
 'categories': [],
 'restrictions': [],
 'userFacingReason': '이 아이디어는 안전하며, 대학가에서 학생들이 필요한 가구와 생활용품을 거래할 수 있는 플랫폼을 제공하는 것입니다.'}

## 08. AI가 이해한 아이디어


In [9]:
display(show_idea_interpretation(idea_context) if idea_context else {'status': 'SKIPPED'})

,항목,AI 이해 결과
0,interpretedProblem,부피가 큰 중고 생활용품은 기존 플랫폼에서 운송 비용 때문에 거래가 어렵다.
1,interpretedTargetUsers,대학가에서 이사·입퇴실을 준비하는 학생과 자취생
2,usageContext,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 거래하는 상황.
3,industryCategory,중고 거래 플랫폼
4,researchScope,대학가 및 학생 커뮤니티
5,conciseIdeaDefinition,대학가 학생들을 위한 지역 기반 중고 거래 서비스
6,targetRegionInterpretation,대학가 지역
7,relevantKnownCompetitorContext,기존의 전국 단위 중고 거래 플랫폼들.


## 09. Readiness / Summary / commitments


In [10]:
display(show_idea_readiness(idea_context) if idea_context else {'status': 'SKIPPED'})

{'readiness': {'status': 'READY_FOR_REVIEW',
  'score': 100,
  'missingFieldKeys': []},
 'readinessDiagnostic': 'CONSISTENT',
 'userFacingSummary': '대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 가까운 사람끼리 거래할 수 있도록 지역 기반 등록·매칭·거래 기능을 제공하는 서비스입니다.',
 'commitmentCandidates': [],
 'contradictions': [],
 'questions': []}

## 10. Seed Analysis


In [11]:
analysis = await engine.analyze_seed(seed) if RUN_STAGED_CORE else None
display(show_seed_analysis(analysis) if analysis else {'status': 'SKIPPED'})

,구분,값
0,탐색 폭,EXPLORE
1,다양성 수용량,5
2,설명,선택 입력 LOCK 0개로 11개 설계 차원이 열려 있습니다. diversityCa...


## 11. Generic Opportunity Kernel


In [12]:
display(analysis.opportunityKernel.model_dump(mode='json') if analysis else {'status': 'SKIPPED'})

{'problemCore': '부피가 큰 중고 생활용품은 기존 플랫폼에서 운송 비용 때문에 거래가 어렵다.',
 'targetCore': '대학가에서 이사·입퇴실을 준비하는 학생과 자취생',
 'useContexts': ['대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 거래하는 상황.'],
 'intentComponents': ['대학가 학생들을 위한 지역 기반 중고 거래 서비스'],
 'mustPreserve': ['부피가 큰 중고 생활용품은 기존 플랫폼에서 운송 비용 때문에 거래가 어렵다.',
  '대학가에서 이사·입퇴실을 준비하는 학생과 자취생',
  '대학가 학생들을 위한 지역 기반 중고 거래 서비스'],
 'maySpecialize': ['핵심 대상의 의미 있는 하위 세그먼트',
  '핵심 사용 맥락의 구체화',
  '가치 제안 또는 offer의 구체화'],
 'forbiddenDriftSummary': '핵심 문제와 대상이 모두 무관한 기회로 교체되면 범위를 벗어납니다.'}

## 12. Design Space


In [13]:
display(show_design_space(analysis) if analysis else {'status': 'SKIPPED'})

,분류,필드,값
0,SOURCE_LOCK,ideaOverview,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 가까운 사람끼리 거...
1,SOURCE_LOCK,problem,부피가 큰 중고 생활용품은 기존 전국 단위 플랫폼에서 운송 비용 때문에 거래가 어렵다.
2,SOURCE_LOCK,targetUsers,대학가에서 이사·입퇴실을 준비하는 학생과 자취생
3,SEMANTIC_ANCHOR,ideaOverview,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 가까운 사람끼리 거...
4,SEMANTIC_ANCHOR,problem,부피가 큰 중고 생활용품은 기존 전국 단위 플랫폼에서 운송 비용 때문에 거래가 어렵다.
5,SEMANTIC_ANCHOR,targetUsers,대학가에서 이사·입퇴실을 준비하는 학생과 자취생
6,OPEN,solutionMechanism,변경 가능
7,OPEN,valueDelivery,변경 가능
8,OPEN,operatingModel,변경 가능
9,OPEN,supplyStructure,변경 가능


## 13. Generate and Adaptively Replenish Plan Pool


In [14]:
plan_validation = (await engine.prepare_portfolio_plans(seed, analysis, max_concepts=MAX_CONCEPTS)
                   if RUN_STAGED_CORE else None)
plans = engine._last_plan_pool if plan_validation else []
print({'totalPlans': len(plans),
       'planningRounds': plan_validation.planningRounds if plan_validation else 0,
       'replenishmentRequested': plan_validation.replenishmentRequested if plan_validation else 0})

{'totalPlans': 12, 'planningRounds': 3, 'replenishmentRequested': 6}


## 14. Plan Count / Adaptive Replenishment Check


In [15]:
display(show_plan_pool_status(engine._last_plan_pool_status) if plan_validation else {'status': 'SKIPPED'})
display({'planningRounds': plan_validation.planningRounds if plan_validation else 0,
         'replenishmentRequested': plan_validation.replenishmentRequested if plan_validation else 0,
         'adaptiveReplenishmentUsed': bool(plan_validation and plan_validation.planningRounds > 1)})

,requestedPoolSize,returnedPoolSize,initialTarget,reserveTarget,reserveAvailable,status
0,7,6,5,2,0,RESERVE_SHORTFALL


{'planningRounds': 3,
 'replenishmentRequested': 6,
 'adaptiveReplenishmentUsed': True}

## 15. Korean Plan Display


In [16]:
display(show_portfolio_plans(plan_validation.acceptedPlans + plan_validation.reservePlans)
        if plan_validation else {'status': 'SKIPPED'})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P1,대학가 중고 거래 플랫폼 A,SELECTED,0.9002,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,거래 중개 · 파트너 네트워크,대학가에서 이사·입퇴실을 준비하는 학생과 자취생,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 거래하는 상황.,학생들이 가까운 지역에서 저렴한 가격으로 중고 생활용품을 쉽게 거래할 수 있도록 지...,대학가 학생들이 필요로 하는 가구와 생활용품을 지역 내에서 손쉽게 거래할 수 있는 ...,지역 기반의 매칭 시스템을 통해 학생들이 서로의 필요를 충족시키는 거래를 가능하게 한다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",대학가 학생들을 위한 특화된 중고 거래 서비스를 제공하여 시장의 니즈를 충족시킨다.
1,P2,대학가 중고 거래 플랫폼 B,SELECTED,0.9002,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,거래 중개 · 파트너 네트워크,대학가에서 이사·입퇴실을 준비하는 학생과 자취생,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 거래하는 상황.,학생들이 필요로 하는 중고 생활용품을 저렴하게 구입하고 판매할 수 있는 기회를 제공한다.,대학가 학생들이 쉽게 접근할 수 있는 중고 거래 플랫폼을 통해 거래의 편리함을 제공한다.,사용자가 직접 등록한 물품을 지역 내 다른 사용자와 연결하여 거래를 성사시킨다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",대학가 학생들을 위한 중고 거래의 편리함을 제공하여 시장의 니즈를 충족시킨다.
2,P3,대학가 중고 거래 플랫폼 C,SELECTED,0.7152,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,거래 중개 · 파트너 네트워크,대학가에서 이사·입퇴실을 준비하는 학생과 자취생,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 거래하는 상황.,학생들이 필요한 중고 생활용품을 저렴하게 거래할 수 있도록 지원한다.,대학가 학생들이 쉽게 접근할 수 있는 중고 거래 플랫폼을 제공한다.,지역 기반의 매칭 시스템을 통해 학생들이 서로의 필요를 충족시키는 거래를 가능하게 한다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",대학가 학생들을 위한 특화된 중고 거래 서비스를 제공하여 시장의 니즈를 충족시킨다.
3,P7,대학가 중고 거래 플랫폼 G,SELECTED,0.6352,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,거래 중개 · 파트너 네트워크,대학가에서 이사·입퇴실을 준비하는 학생과 자취생,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 거래하는 상황.,학생들이 필요한 중고 생활용품을 저렴하게 구입하고 판매할 수 있는 기회를 제공한다.,대학가 학생들이 필요로 하는 가구와 생활용품을 지역 내에서 손쉽게 거래할 수 있는 ...,지역 기반의 매칭 시스템을 통해 학생들이 서로의 필요를 충족시키는 거래를 가능하게 한다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",대학가 학생들을 위한 특화된 중고 거래 서비스를 제공하여 시장의 니즈를 충족시킨다.


## 16. Plan Lock/Intent Validation


In [17]:
display({'accepted': [p.planId for p in plan_validation.acceptedPlans] if plan_validation else [],
         'rejected': [p.model_dump(mode='json') for p in plan_validation.rejectedPlans]
                     if plan_validation else []})

{'accepted': ['P1', 'P2', 'P3', 'P7'],
 'rejected': [{'planId': 'P4',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P1'},
  {'planId': 'P5',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P2'},
  {'planId': 'P6',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P1'},
  {'planId': 'P8',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P7'},
  {'planId': 'P9',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P7'},
  {'planId': 'P10',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P7'},
  {'planId': 'P11',
   'reasonCode':

## 17. Portfolio Family / Variant / Distinct


In [18]:
display(show_plan_diversity(plan_validation.diversity) if plan_validation else {'status': 'SKIPPED'})

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,P1,P2,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
1,P1,P3,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
2,P2,P3,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, solutionThesis",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
3,P1,P4,DUPLICATE,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...",,THESIS_AND_ARCHITECTURE,False,이름이나 표현을 제외한 Concept Thesis와 Business Architec...
4,P1,P5,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
5,P2,P5,DUPLICATE,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...",,THESIS_AND_ARCHITECTURE,False,이름이나 표현을 제외한 Concept Thesis와 Business Architec...
6,P1,P6,DUPLICATE,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...",,THESIS_AND_ARCHITECTURE,False,이름이나 표현을 제외한 Concept Thesis와 Business Architec...
7,P1,P7,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...",valuePropositionThesis,MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
8,P2,P7,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","offerThesis, solutionThesis",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
9,P3,P7,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...


## 18. Selected + Reserve Plans


In [19]:
selected_plans = plan_validation.acceptedPlans if plan_validation else []
reserve_plans = plan_validation.reservePlans if plan_validation else []
display(show_portfolio_plans(selected_plans + reserve_plans))
display({'selected': [p.planId for p in selected_plans], 'reserve': [p.planId for p in reserve_plans]})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P1,대학가 중고 거래 플랫폼 A,SELECTED,0.9002,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,거래 중개 · 파트너 네트워크,대학가에서 이사·입퇴실을 준비하는 학생과 자취생,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 거래하는 상황.,학생들이 가까운 지역에서 저렴한 가격으로 중고 생활용품을 쉽게 거래할 수 있도록 지...,대학가 학생들이 필요로 하는 가구와 생활용품을 지역 내에서 손쉽게 거래할 수 있는 ...,지역 기반의 매칭 시스템을 통해 학생들이 서로의 필요를 충족시키는 거래를 가능하게 한다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",대학가 학생들을 위한 특화된 중고 거래 서비스를 제공하여 시장의 니즈를 충족시킨다.
1,P2,대학가 중고 거래 플랫폼 B,SELECTED,0.9002,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,거래 중개 · 파트너 네트워크,대학가에서 이사·입퇴실을 준비하는 학생과 자취생,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 거래하는 상황.,학생들이 필요로 하는 중고 생활용품을 저렴하게 구입하고 판매할 수 있는 기회를 제공한다.,대학가 학생들이 쉽게 접근할 수 있는 중고 거래 플랫폼을 통해 거래의 편리함을 제공한다.,사용자가 직접 등록한 물품을 지역 내 다른 사용자와 연결하여 거래를 성사시킨다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",대학가 학생들을 위한 중고 거래의 편리함을 제공하여 시장의 니즈를 충족시킨다.
2,P3,대학가 중고 거래 플랫폼 C,SELECTED,0.7152,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,거래 중개 · 파트너 네트워크,대학가에서 이사·입퇴실을 준비하는 학생과 자취생,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 거래하는 상황.,학생들이 필요한 중고 생활용품을 저렴하게 거래할 수 있도록 지원한다.,대학가 학생들이 쉽게 접근할 수 있는 중고 거래 플랫폼을 제공한다.,지역 기반의 매칭 시스템을 통해 학생들이 서로의 필요를 충족시키는 거래를 가능하게 한다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",대학가 학생들을 위한 특화된 중고 거래 서비스를 제공하여 시장의 니즈를 충족시킨다.
3,P7,대학가 중고 거래 플랫폼 G,SELECTED,0.6352,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,거래 중개 · 파트너 네트워크,대학가에서 이사·입퇴실을 준비하는 학생과 자취생,대학가에서 이사하거나 기숙사를 옮기는 학생들이 가구와 생활용품을 거래하는 상황.,학생들이 필요한 중고 생활용품을 저렴하게 구입하고 판매할 수 있는 기회를 제공한다.,대학가 학생들이 필요로 하는 가구와 생활용품을 지역 내에서 손쉽게 거래할 수 있는 ...,지역 기반의 매칭 시스템을 통해 학생들이 서로의 필요를 충족시키는 거래를 가능하게 한다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",대학가 학생들을 위한 특화된 중고 거래 서비스를 제공하여 시장의 니즈를 충족시킨다.


{'selected': ['P1', 'P2', 'P3', 'P7'], 'reserve': []}

## 19. Candidate 1


In [20]:
candidate_one = (await engine.expand_plan(seed, selected_plans[0], 1)
                 if RUN_STAGED_CORE and selected_plans else None)
display(show_candidates([candidate_one]) if candidate_one else [])

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,대학가 중고 거래 플랫폼 A,지역 기반의 매칭 시스템을 통해 학생들이 서로의 필요를 충족시키는 거래를 가능하게 한다.,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '대학가에서 이사·입...,프리미엄 서비스 및 광고를 통해 수익을 창출한다.,학생 커뮤니티와의 협력을 통해 플랫폼을 운영하고 홍보한다.


## 20. Candidate 1 Korean/Governance


In [21]:
candidate_one_reports = []
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'status': 'PENDING_FULL_CANDIDATE_RECOVERY'})

{'candidateId': 'C1', 'status': 'PENDING_FULL_CANDIDATE_RECOVERY'}

## 21. Candidate 1 Actual Generic Descriptor


In [22]:
display(show_concept_descriptors([candidate_one]) if candidate_one else [])

,entityId,family,dimension,code,confidence,source
0,C1,INTERMEDIARY:PARTNER_NETWORK,businessRole,INTERMEDIARY,HIGH,RULE
1,C1,INTERMEDIARY:PARTNER_NETWORK,operatingModel,PARTNER_NETWORK,HIGH,RULE
2,C1,INTERMEDIARY:PARTNER_NETWORK,partnerModel,PARTNER_NETWORK,HIGH,RULE
3,C1,INTERMEDIARY:PARTNER_NETWORK,deliveryModel,PHYSICAL_DELIVERY,HIGH,RULE
4,C1,INTERMEDIARY:PARTNER_NETWORK,transactionModel,OTHER,LOW,UNKNOWN
5,C1,INTERMEDIARY:PARTNER_NETWORK,monetizationModel,OTHER,LOW,UNKNOWN
6,C1,INTERMEDIARY:PARTNER_NETWORK,customerInteractionModel,OTHER,LOW,UNKNOWN
7,C1,INTERMEDIARY:PARTNER_NETWORK,dataDependency,NONE,NaN,NaN
8,C1,INTERMEDIARY:PARTNER_NETWORK,physicalDependency,MATERIAL,NaN,NaN


## 22. Candidate 1 Fidelity


In [23]:
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'})

{'candidateId': 'C1',
 'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'}

## 23. Remaining Candidates


In [24]:
remaining_candidates = []
if RUN_STAGED_CORE:
    for i, plan in enumerate(selected_plans[1:], 2):
        remaining_candidates.append(await engine.expand_plan(seed, plan, i))
candidate_drafts = ([candidate_one] if candidate_one else []) + remaining_candidates
display(show_candidates(candidate_drafts))

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,대학가 중고 거래 플랫폼 A,지역 기반의 매칭 시스템을 통해 학생들이 서로의 필요를 충족시키는 거래를 가능하게 한다.,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '대학가에서 이사·입...,프리미엄 서비스 및 광고를 통해 수익을 창출한다.,학생 커뮤니티와의 협력을 통해 플랫폼을 운영하고 홍보한다.
1,C2,L2,None,대학가 중고 거래 플랫폼 B,사용자가 직접 등록한 물품을 지역 내 다른 사용자와 연결하여 거래를 성사시킨다.,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '대학가에서 이사·입...,프리미엄 서비스 및 광고를 통해 수익을 창출한다.,대학가 내 커뮤니티와 협력하여 플랫폼을 운영한다.
2,C3,L3,None,대학가 중고 거래 플랫폼 C,지역 기반의 매칭 시스템을 통해 학생들이 서로의 필요를 충족시키는 거래를 가능하게 한다.,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '대학가에서 이사·입...,프리미엄 서비스 및 광고를 통해 수익을 창출한다.,학생 커뮤니티와의 협력을 통해 플랫폼을 운영하고 홍보한다.
3,C4,L4,None,대학가 중고 거래 플랫폼 G,지역 기반의 매칭 시스템을 통해 학생들이 서로의 필요를 충족시키는 거래를 가능하게 한다.,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '대학가에서 이사·입...,프리미엄 서비스 및 광고를 통해 수익을 창출한다.,학생 커뮤니티와의 협력을 통해 플랫폼을 운영하고 홍보한다.


## 24. Candidate Actual Generic Descriptors


In [25]:
display(show_concept_descriptors(candidate_drafts))

,entityId,family,dimension,code,confidence,source
0,C1,INTERMEDIARY:PARTNER_NETWORK,businessRole,INTERMEDIARY,HIGH,RULE
1,C1,INTERMEDIARY:PARTNER_NETWORK,operatingModel,PARTNER_NETWORK,HIGH,RULE
2,C1,INTERMEDIARY:PARTNER_NETWORK,partnerModel,PARTNER_NETWORK,HIGH,RULE
3,C1,INTERMEDIARY:PARTNER_NETWORK,deliveryModel,PHYSICAL_DELIVERY,HIGH,RULE
4,C1,INTERMEDIARY:PARTNER_NETWORK,transactionModel,OTHER,LOW,UNKNOWN
5,C1,INTERMEDIARY:PARTNER_NETWORK,monetizationModel,OTHER,LOW,UNKNOWN
6,C1,INTERMEDIARY:PARTNER_NETWORK,customerInteractionModel,OTHER,LOW,UNKNOWN
7,C1,INTERMEDIARY:PARTNER_NETWORK,dataDependency,NONE,NaN,NaN
8,C1,INTERMEDIARY:PARTNER_NETWORK,physicalDependency,MATERIAL,NaN,NaN
9,C2,INTERMEDIARY:PARTNER_NETWORK,businessRole,INTERMEDIARY,HIGH,RULE


## 25. Candidate Recovery / Portfolio Relations


In [26]:
candidate_preparation = (await engine.prepare_candidate_portfolio(
    seed, plan_validation, max_concepts=MAX_CONCEPTS, initial_candidates=candidate_drafts)
    if RUN_STAGED_CORE and plan_validation else None)
candidates = candidate_preparation.candidates if candidate_preparation else []
candidate_reports = candidate_preparation.reports if candidate_preparation else []
display(show_candidate_recovery(candidate_preparation) if candidate_preparation else {'status': 'SKIPPED'})
candidate_pairwise = [engine.compare_candidates(candidates[i], candidates[j])
                      for i in range(len(candidates)) for j in range(i + 1, len(candidates))]
display(show_plan_diversity(candidate_pairwise))

{'summary':    candidateGenerated  candidateAcceptedInitially  candidateRegenerated  \
 0                   4                           4                     0   
 
    candidateRecovered  reservePlansActivated  candidateRecoveryReplans  \
 0                   0                      0                         2   
 
    finalCandidatePortfolio  
 0                        4  ,
 'attempts':   candidateId  schemaValid  hardLockPreserved  semanticAnchorPreserved  \
 0          C1         True               True                     True   
 1          C2         True               True                     True   
 2          C3         True               True                     True   
 3          C4         True               True                     True   
 
    planFidelity anchorDecision fidelityDecision  contentLanguageValid  \
 0          True           PASS          ADAPTED                  True   
 1          True           PASS          ADAPTED                  True   
 2         

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,C1,C2,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
1,C1,C3,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...",valuePropositionThesis,MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
2,C1,C4,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
3,C2,C3,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
4,C2,C4,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","offerThesis, solutionThesis",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
5,C3,C4,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...


## 26. Legal Fact Completeness + Business Design Completion + C1 Fact Pattern


In [27]:
prechecks = [engine.legal_precheck(item) for item in candidates] if RUN_STAGED_LEGAL else []
display(show_legal_precheck(prechecks))
legal_preparation = (await engine.prepare_legal_candidates(seed, candidates)
                     if RUN_STAGED_LEGAL and candidates else None)
candidates_before_legal = candidates
candidates = legal_preparation.candidates if legal_preparation else candidates
display({'factCompleteness': [item.model_dump(mode='json') for item in legal_preparation.reports]
                              if legal_preparation else [],
         'roleSemanticBatchCalls': legal_preparation.roleSemanticBatchCalls if legal_preparation else 0,
         'dependencySemanticBatchCalls': legal_preparation.dependencySemanticBatchCalls if legal_preparation else 0,
         'completionAttempted': legal_preparation.completionAttempted if legal_preparation else 0,
         'completionValidated': legal_preparation.completionValidated if legal_preparation else 0,
         'completionAccepted': legal_preparation.completionAccepted if legal_preparation else 0,
         'completionExhausted': legal_preparation.completionExhausted if legal_preparation else 0,
         'completionCompliance': [item.model_dump(mode='json') for item in legal_preparation.completionCompliance] if legal_preparation else [],
         'factConsistency': [item.model_dump(mode='json') for item in legal_preparation.consistencyReports] if legal_preparation else [],
         'consistencyRepairAttempted': legal_preparation.consistencyRepairAttempted if legal_preparation else 0,
         'consistencyRepairAccepted': legal_preparation.consistencyRepairAccepted if legal_preparation else 0,
         'consistencyRepairExhausted': legal_preparation.consistencyRepairExhausted if legal_preparation else 0,
         'preLegalExclusions': legal_preparation.excludedCandidates if legal_preparation else []})
display(show_legal_fact_pattern(candidates[0].candidate, seed) if candidates else [])

,candidateId,label,directSeller,intermediary,regulatedPhysicalActivity,personalDataDependency,qualificationDependency,riskHints
0,C1,Structural risk precheck — not final legal review,False,False,True,False,False,[물리 활동]
1,C2,Structural risk precheck — not final legal review,True,False,True,False,False,[물리 활동]
2,C3,Structural risk precheck — not final legal review,True,False,True,False,False,[물리 활동]
3,C4,Structural risk precheck — not final legal review,True,True,True,False,False,[물리 활동]


{'factCompleteness': [{'candidateId': 'C1',
   'status': 'COMPLETE',
   'missingDesignFacts': [],
   'contradictions': [],
   'completionRequirements': [],
   'affectedFields': [],
   'roleSemantics': [{'field': 'platformRole',
     'status': 'MATCH',
     'presence': 'PRESENT',
     'deterministicStatus': 'MATCH',
     'semanticUsed': False,
     'semanticStatus': 'NOT_RUN',
     'finalStatus': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'providerRole',
     'status': 'MATCH',
     'presence': 'PRESENT',
     'deterministicStatus': 'AMBIGUOUS',
     'semanticUsed': True,
     'semanticStatus': 'MATCH',
     'finalStatus': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'sellerRole',
     'status': 'MATCH',
     'presence': 'PRESENT',
     'deterministicStatus': 'AMBIGUOUS',
     'semanticUsed': True,
     'semanticStatus': 'MATCH',
     'finalStatus': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'intermediaryRole',
     'status': 

,field,legalFact,source,authority,decision
0,platformRole,학생 커뮤니티와의 협력을 통해 플랫폼을 운영하고 홍보한다.,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
1,providerRole,플랫폼 운영자,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
2,sellerRole,학생 사용자,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
3,intermediaryRole,없음,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
4,transactionFlow,"[학생 사용자가 물품을 등록한다., 다른 학생 사용자가 해당 물품을 검색하고 연락한...",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
5,paymentFlow,"[학생 사용자가 거래 금액을 결제한다., 플랫폼이 수수료를 차감한 후 판매자에게 금...",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
6,personalDataUsage,[],CONCEPT_GENERATED,REVIEWABLE,PROPOSED
7,physicalActivities,[배송 서비스 연계],CONCEPT_GENERATED,REVIEWABLE,PROPOSED
8,partnerRequirements,[],CONCEPT_GENERATED,REVIEWABLE,PROPOSED
9,qualificationRequirements,[],CONCEPT_GENERATED,REVIEWABLE,PROPOSED


## 27. Prepared Legal C1 Evidence Summary


In [28]:
legal_adapter = CurrentLegalAdapter() if RUN_STAGED_LEGAL else None
legal_c1_input = (legal_adapter.task_input(candidates[0].candidate, seed)
                  if legal_adapter and candidates else None)
display({'candidateId': candidates[0].candidateId if candidates else None,
         'externalFacts': legal_c1_input['externalFactContext']['facts'] if legal_c1_input else [],
         'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'})

{'candidateId': 'C1',
 'externalFacts': [],
 'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'}

## 28. Full Evidence Judgment — C1 Staged Smoke


In [29]:
RUN_FULL_LEGAL_C1 = RUN_STAGED_LEGAL
legal_one = None
if RUN_FULL_LEGAL_C1 and candidates:
    try:
        legal_one = (await engine.review_legal(seed, candidates[:1]))[0]
        display(show_legal_result([legal_one]))
    except Exception:
        display(show_legal_failure(candidates[0].candidateId, engine.gateway))
else:
    print('SKIPPED — RUN_FULL_LEGAL_C1=True로 명시해야 실행됩니다.')

,candidateId,route,productionStatus,sourceStatus,evidenceCoverage,reviewPhase,factCompletenessStatus,legalSourceStatus,finalEvidenceJudgmentExecuted,recoveryResolution,...,legalClarificationCount,safeSummary,unknownFacts,requiredControls,requiredPartnersAndQualifications,requiredDisclosures,prohibitedVariants,evidenceCount,evidenceRefs,evidenceDiagnostics
0,C1,NEEDS_INPUT,REDESIGNABLE,SOURCE_COMPLETE,None,LEGAL_SOURCE,None,SOURCE_COMPLETE,False,LEGAL_FACT_REQUIRED_NOT_REDESIGN,...,0,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,[What specific advertising claims will be made...,[],[],[],[],16,"[{'referenceIndex': 0, 'sourceType': 'OFFICIAL...","{'coverageStatus': 'SOURCE_COMPLETE', 'coverag..."


## 29. C1 Route + Staged Redesign/Compliance/Second Legal


In [30]:
c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = ([], [], [], 0, 0)
if legal_one and candidates:
    c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[:1], [legal_one])
display({'initialRoute': legal_one.route.value if legal_one else 'SKIPPED',
         'redesignRequirements': legal_one.redesignRequirements if legal_one else [],
         'recoveryReviews': [item.model_dump(mode='json') for item in c1_legal_all[1:]],
         'requiredInputs': c1_required_inputs, 'terminalCandidates': len(c1_portfolio),
         'redesigned': c1_redesigned, 'replanned': c1_replanned})

{'initialRoute': 'NEEDS_INPUT',
 'redesignRequirements': ['What specific advertising claims will be made on the platform?'],
 'recoveryReviews': [],
 'requiredInputs': [{'candidateId': 'C1',
   'scope': 'CANDIDATE',
   'unknownFacts': ['What specific advertising claims will be made on the platform?'],
   'conflictingLock': None,
   'currentValue': None,
   'requiredLegalChange': None,
   'reason': '법률검토 전에 Concept의 사업 구조를 보완해야 합니다.',
   'possibleUserAction': '현재 운영 계획에서 이 요구와 관련된 실제 사업 사실을 확인해 주십시오.',
   'question': '현재 운영 계획에서 이 요구와 관련된 실제 사업 사실을 확인해 주십시오.',
   'safeSummary': '법률검토 전에 Concept의 사업 구조를 보완해야 합니다.'}],
 'terminalCandidates': 0,
 'redesigned': 0,
 'replanned': 0}

## 30. Remaining 4 Legal + Exhaustive Recovery Summary


In [31]:
RUN_REMAINING_LEGAL = RUN_STAGED_FULL
legal_remaining = []
portfolio, legal_all, required_inputs = (list(c1_portfolio), list(c1_legal_all), list(c1_required_inputs))
redesigned_count, replanned_count = c1_redesigned, c1_replanned
c1_terminal = bool(c1_portfolio or c1_required_inputs or (c1_legal_all and c1_legal_all[-1].route.value == 'SYSTEM_FAILURE'))
if RUN_REMAINING_LEGAL and c1_terminal and len(candidates) > 1:
    legal_remaining = await engine.review_legal(seed, candidates[1:])
    rest_portfolio, rest_legal, rest_inputs, rest_redesigned, rest_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[1:], legal_remaining)
    portfolio += rest_portfolio; legal_all += rest_legal; required_inputs += rest_inputs
    redesigned_count += rest_redesigned; replanned_count += rest_replanned
else:
    print('SKIPPED — C1이 정상 terminal에 도달한 후 remaining Legal을 실행합니다.')
legal_initial = ([legal_one] if legal_one else []) + legal_remaining
display({'recoveryTrace': [item.model_dump(mode='json') for item in legal_all
                           if item.candidateId not in {x.candidateId for x in legal_initial}],
         'requiredInputs': required_inputs, 'metrics': engine._legal_metrics})
print({'Plan Selected': len(selected_plans),
       'Candidate Generated': candidate_preparation.candidateGenerated if candidate_preparation else 0,
       'Candidate Valid Initially': candidate_preparation.candidateAcceptedInitially if candidate_preparation else 0,
       'Candidate Regenerated': candidate_preparation.candidateRegenerated if candidate_preparation else 0,
       'Candidate Recovered': candidate_preparation.candidateRecovered if candidate_preparation else 0,
       'Fact Completion Attempted': legal_preparation.completionAttempted if legal_preparation else 0,
       'Fact Completion Validated': legal_preparation.completionValidated if legal_preparation else 0,
       'Fact Completion Accepted': legal_preparation.completionAccepted if legal_preparation else 0,
       'Dependency Semantic Calls': legal_preparation.dependencySemanticBatchCalls if legal_preparation else 0,
       'Completion Compliance PASS': sum(item.status == 'PASS' for item in legal_preparation.completionCompliance) if legal_preparation else 0,
       'Fact Consistency Invalid': sum(item.status == 'INVALID_FACT' for item in legal_preparation.consistencyReports) if legal_preparation else 0,
       'Consistency Repair Accepted': legal_preparation.consistencyRepairAccepted if legal_preparation else 0,
       'Legal Ready': len(candidates) if legal_preparation else 0,
       'Legal Reviewed': len(legal_initial),
       'Legal Accepted': sum(item.route.value == 'ACCEPT' for item in legal_all),
       'Legal Redesigned': redesigned_count, 'Legal Replanned': replanned_count,
       'Final Portfolio': len(portfolio)})

{'recoveryTrace': [],
 'requiredInputs': [{'candidateId': 'C1',
   'scope': 'CANDIDATE',
   'unknownFacts': ['What specific advertising claims will be made on the platform?'],
   'conflictingLock': None,
   'currentValue': None,
   'requiredLegalChange': None,
   'reason': '법률검토 전에 Concept의 사업 구조를 보완해야 합니다.',
   'possibleUserAction': '현재 운영 계획에서 이 요구와 관련된 실제 사업 사실을 확인해 주십시오.',
   'question': '현재 운영 계획에서 이 요구와 관련된 실제 사업 사실을 확인해 주십시오.',
   'safeSummary': '법률검토 전에 Concept의 사업 구조를 보완해야 합니다.'},
  {'candidateId': 'C2',
   'scope': 'CANDIDATE',
   'unknownFacts': ['What specific types of products will be sold on the platform?'],
   'conflictingLock': None,
   'currentValue': None,
   'requiredLegalChange': None,
   'reason': '법률검토 전에 Concept의 사업 구조를 보완해야 합니다.',
   'possibleUserAction': '현재 운영 계획에서 이 요구와 관련된 실제 사업 사실을 확인해 주십시오.',
   'question': '현재 운영 계획에서 이 요구와 관련된 실제 사업 사실을 확인해 주십시오.',
   'safeSummary': '법률검토 전에 Concept의 사업 구조를 보완해야 합니다.'},
  {'candidateId': 'C3',
   'scope': 'CANDIDATE',
  

{'Plan Selected': 4, 'Candidate Generated': 4, 'Candidate Valid Initially': 4, 'Candidate Regenerated': 0, 'Candidate Recovered': 0, 'Fact Completion Attempted': 0, 'Fact Completion Validated': 0, 'Fact Completion Accepted': 0, 'Dependency Semantic Calls': 1, 'Completion Compliance PASS': 0, 'Fact Consistency Invalid': 0, 'Consistency Repair Accepted': 0, 'Legal Ready': 4, 'Legal Reviewed': 4, 'Legal Accepted': 0, 'Legal Redesigned': 0, 'Legal Replanned': 0, 'Final Portfolio': 0}


## 31. Replan


In [32]:
display(show_replan(type('PortfolioView', (), {'concepts': portfolio})()))
print({'replanned': replanned_count, 'reserveAvailable': len(reserve_plans)})

""


{'replanned': 0, 'reserveAvailable': 0}


## 32. Final Portfolio


In [33]:
legal_terminal_status = ('READY_FULL' if len(portfolio) == MAX_CONCEPTS else
    'READY_LIMITED' if portfolio else
    'LEGAL_RECOVERY_COMPLETE_NO_ACCEPTED_CANDIDATE' if legal_initial and len(legal_initial) == len(candidates)
    else 'LEGAL_PENDING')
display(show_final_portfolio(type('PortfolioView', (), {'concepts': portfolio})()) if portfolio else {'status': legal_terminal_status})

{'status': 'LEGAL_RECOVERY_COMPLETE_NO_ACCEPTED_CANDIDATE'}

## 33. Unresolved Candidate Summary


In [34]:
display(show_required_inputs(required_inputs) if required_inputs else {'unresolved': []})

,candidateId,scope,unknownFacts,reason,possibleUserAction,currentValue,requiredLegalChange,safeSummary
0,C1,CANDIDATE,[What specific advertising claims will be made...,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,현재 운영 계획에서 이 요구와 관련된 실제 사업 사실을 확인해 주십시오.,None,None,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.
1,C2,CANDIDATE,[What specific types of products will be sold ...,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,현재 운영 계획에서 이 요구와 관련된 실제 사업 사실을 확인해 주십시오.,None,None,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.
2,C3,CANDIDATE,[What specific personal data will be collected...,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,현재 운영 계획에서 이 요구와 관련된 실제 사업 사실을 확인해 주십시오.,None,None,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.
3,C4,CANDIDATE,[What specific advertising claims will be made...,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,현재 운영 계획에서 이 요구와 관련된 실제 사업 사실을 확인해 주십시오.,None,None,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.


## 34. Manual Concept Selection


In [35]:
SELECTED_CANDIDATE_ID = portfolio[0].candidateId if portfolio else None  # 사용자가 수정
selected_concept = next((item for item in portfolio if item.candidateId == SELECTED_CANDIDATE_ID), None)
print({'selectedCandidateId': SELECTED_CANDIDATE_ID})

{'selectedCandidateId': None}


## 35. 7 Hypotheses


In [36]:
hypotheses = (engine.build_or_load_current_hypothesis_contract(selected_concept)
              if RUN_STAGED_FULL and selected_concept else [])
hypotheses = await engine.resolve_hypothesis_semantics(hypotheses) if hypotheses else []
display(show_hypotheses(hypotheses))
display(show_hypothesis_readiness(hypotheses))

""


{'All Values Semantically Valid': False,
 'All Decisions Confirmed': False,
 'Ready For Handoff': False,
 'status': 'NOT_READY',
 'reason': 'NO_SELECTED_CONCEPT_OR_HYPOTHESES',
 'unresolvedHypotheses': []}

## 36. Confirm / Edit


In [37]:
CONFIRM_ALL_PROPOSED = True
HYPOTHESIS_EDITS = {
    # 'PRICE': '월 17,900원',
}
confirmed_hypotheses = engine.confirm_hypotheses(
    hypotheses, HYPOTHESIS_EDITS, confirm_all_proposed=CONFIRM_ALL_PROPOSED) if hypotheses else []
display(show_hypotheses(confirmed_hypotheses))
hypothesis_readiness = show_hypothesis_readiness(confirmed_hypotheses)
display(hypothesis_readiness)

""


{'All Values Semantically Valid': False,
 'All Decisions Confirmed': False,
 'Ready For Handoff': False,
 'status': 'NOT_READY',
 'reason': 'NO_SELECTED_CONCEPT_OR_HYPOTHESES',
 'unresolvedHypotheses': []}

## 37. Actual Delta Legal


In [38]:
RUN_DELTA_LEGAL = True
delta_legal_result = None
if RUN_STAGED_FULL and RUN_DELTA_LEGAL and selected_concept and any(h.deltaLegalRequired for h in confirmed_hypotheses):
    delta_legal_result = await engine.review_delta_legal(seed, selected_concept, confirmed_hypotheses)
    confirmed_hypotheses = engine.mark_delta_legal_reviewed(confirmed_hypotheses, delta_legal_result)
display(delta_legal_result.model_dump(mode='json') if delta_legal_result else {'status': 'NOT_REQUIRED_OR_SKIPPED'})

{'status': 'NOT_REQUIRED_OR_SKIPPED'}

## 38. Market Seed


In [39]:
handoff = None
if selected_concept and legal_all and hypothesis_readiness['Ready For Handoff']:
    handoff = engine.build_downstream_handoff(seed, selected_concept, confirmed_hypotheses, legal_all)
display(handoff.marketAnalysisSeedSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'status': 'NOT_READY',
 'reason': 'NO_SELECTED_CONCEPT_OR_HYPOTHESES',
 'unresolvedHypotheses': []}

## 39. Marketing Source


In [40]:
display(handoff.marketingSourceSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'status': 'NOT_READY',
 'reason': 'NO_SELECTED_CONCEPT_OR_HYPOTHESES',
 'unresolvedHypotheses': []}

## 40. Contract Compatibility


In [41]:
display(show_downstream_handoff(handoff) if handoff else {
    'contract': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'contract': 'NOT_READY',
 'reason': 'NO_SELECTED_CONCEPT_OR_HYPOTHESES',
 'unresolvedHypotheses': []}

## 41. Trace


In [42]:
display(show_trace(engine.trace))

,순서,시각,stage,action,entity,parent,status,mode,호출,요약,decision,reasonCode
0,1,2026-08-10T08:44:04.196747+00:00,CREATED,CREATED,NaN,NaN,RUNNING,LIVE,NaN,V2 Lab 실행을 생성했습니다.,NaN,NaN
1,2,2026-08-10T08:44:09.859335+00:00,SAFETY_CHECKING,IDEA_BRIEF_DERIVED,NaN,NaN,PASS,LIVE,1.0,Idea interpretation/readiness를 보존했습니다: READY_F...,NaN,NaN
2,3,2026-08-10T08:44:09.926717+00:00,SEED_ANALYZING,ANALYZED,lab-idea-brief,NaN,PASS,LIVE,NaN,필수 3개와 LOCK 3개를 분류했습니다.,NaN,NaN
3,4,2026-08-10T08:44:09.927347+00:00,SEED_ANALYZING,DESIGN_SPACE_READY,NaN,NaN,PASS,LIVE,NaN,Open=11 Constrained=0 Breadth=EXPLORE,NaN,NaN
4,5,2026-08-10T08:44:10.006283+00:00,PLANNING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,최대 5개 동적 plan을 요청합니다.,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
92,93,2026-08-10T08:48:32.613320+00:00,LEGAL_REVIEWING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,공식 Legal contract route를 실행합니다.,NaN,NaN
93,94,2026-08-10T08:49:01.462061+00:00,LEGAL_REVIEWING,REVIEWED,C2,NaN,NEEDS_INPUT,LIVE,20.0,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,NEEDS_INPUT,NaN
94,95,2026-08-10T08:49:23.709997+00:00,LEGAL_REVIEWING,REVIEWED,C3,NaN,NEEDS_INPUT,LIVE,21.0,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,NEEDS_INPUT,NaN
95,96,2026-08-10T08:49:38.747278+00:00,LEGAL_REVIEWING,REVIEWED,C4,NaN,NEEDS_INPUT,LIVE,22.0,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,NEEDS_INPUT,NaN


## 42. Provider/Legal Usage


In [43]:
display(show_provider_usage(engine.gateway.usage))
print('상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.')

,논리 작업,상위 외부 작업,논리 stage별,상위 외부 작업 stage별,재시도,소요(ms),모드별,token,보고 비용
0,22,22,"{'SAFETY_CHECKING': 1, 'PLANNING': 5, 'NORMALI...","{'SAFETY_CHECKING': 1, 'PLANNING': 5, 'NORMALI...",0,333314,{'LIVE': 22},None,None


상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.


## 43. Replay Manifest


In [44]:
display(show_replay_manifest(engine.gateway))

{'status': 'REPLAY_PARTIAL',
 'entries':                    operation  \
 0               LEGAL_REVIEW   
 1          SEMANTIC_RELATION   
 2              PLAN_FIDELITY   
 3                     EXPAND   
 4    NORMALIZE_ARCHITECTURES   
 ..                       ...   
 637    IDEA_BRIEF_DERIVATION   
 638       PLAN_REPLENISHMENT   
 639                PLAN_POOL   
 640  NORMALIZE_ARCHITECTURES   
 641             LEGAL_REVIEW   
 
                                                   hash operationVersion  \
 0    00489dd62bff3c4b9e8711093fb3e398a39dde1547de32...               v3   
 1    00c1eeb4f93989a6db3891fb7a22355a71a68800a6f641...               v3   
 2    00e62d4c1a15307c38d9024cb66e28f8d13300af644fff...               v3   
 3    0120f755ef50e68db6e1d7d1adb0563892b2f762578608...               v3   
 4    013ebfc11d61a317c1041ddcf75902b1dab0d310e73666...               v1   
 ..                                                 ...              ...   
 637  fc5b3fe1b2fb09f8b0b6587a

## 44. One-click MOCK


In [45]:
mock_result = None
if LIVE_TEST_LEVEL == 'FULL_E2E' and MODE != 'LIVE':
    mock_result = await ConceptPortfolioEngine('MOCK').run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=True)
display(show_run_summary(mock_result) if mock_result else {'status': 'SKIPPED'})
assert mock_result is None or (mock_result.handoff and mock_result.handoff.contractStatus == 'CONTRACT_PASS')

{'status': 'SKIPPED'}

## 45. One-click REPLAY


In [46]:
RUN_ONE_CLICK_REPLAY = False
replay_result = None
if RUN_ONE_CLICK_REPLAY:
    replay_gateway = ProviderGateway('REPLAY', recordings_dir=RECORDINGS_DIR)
    replay_result = await ConceptPortfolioEngine('REPLAY', gateway=replay_gateway).run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=False)
display(show_run_summary(replay_result) if replay_result else {'status': 'SKIPPED'})

{'status': 'SKIPPED'}

## 46. One-click LIVE


In [47]:
RUN_ONE_CLICK_LIVE = True
live_result = None
if RUN_ONE_CLICK_LIVE and LIVE_TEST_LEVEL == 'ONE_CLICK':
    assert MODE == 'LIVE', 'MODE=LIVE를 먼저 명시하세요.'
    one_click_gateway = ProviderGateway('LIVE', recordings_dir=RECORDINGS_DIR)
    one_click_engine = ConceptPortfolioEngine('LIVE', gateway=one_click_gateway)
    live_result = await one_click_engine.run_full(TEST_INPUT, max_concepts=MAX_CONCEPTS,
                                                  auto_confirm_hypotheses=False)
display(show_run_summary(live_result) if live_result else {'status': 'SKIPPED'})
if live_result:
    display(show_live_validation_summary(LIVE_SCENARIO, live_result))
    display(show_required_inputs(live_result))
    display(show_pre_legal_exclusions(live_result))
    display(show_legal_resolutions(live_result))
    if live_result.runStatus.value == 'FAILED':
        display(show_run_failure(live_result))
        display(show_provider_failure(one_click_engine.gateway))
        display(show_provider_usage(live_result.providerUsage))
        display(show_trace(live_result.trace[-20:]))
        display({'unresolvedCandidates': live_result.unresolvedCandidates,
                 'lastSuccessfulStage': live_result.failureDiagnostics.lastSuccessfulStage if live_result.failureDiagnostics else None,
                 'firstFailedStage': live_result.failureDiagnostics.firstFailedStage if live_result.failureDiagnostics else None})

{'status': 'SKIPPED'}